# 🧪 A/B Testing: Promotional Voucher Optimization
### Two-Proportion Z-Test on Checkout Completion Rates
**Author:** Senior Data Analyst & Analytics Engineer  
**Experiment:** Fixed Discount Voucher (Group A) vs. Percentage Discount Voucher (Group B)  
**Objective:** Determine whether offering a percentage-based discount (e.g., 15% OFF) yields a statistically significant increase in Order Completion Rate compared to a fixed VND voucher (e.g., 50,000 VND OFF) at the 95% Confidence Level ($\alpha = 0.05$).

---

In [ ]:
# Step 1: Import Statistical & Data Visualization Packages
import numpy as np
import pandas as pd
import scipy.stats as stats
from statsmodels.stats.proportion import proportions_ztest, proportion_confint
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 5)
print("A/B Testing environment initialized.")

## 1. Experiment Formulation & Hypothesis Definition

### **Business Context**
SmartSale is optimizing coupon campaign performance across promotional banners and checkout modals.
- **Control Group (A):** Users presented with a **Fixed Cash Voucher** (e.g., `SAVE50K` - 50.000₫ OFF).
- **Variant Group (B):** Users presented with a **Percentage Discount Voucher** (e.g., `DISC15PCT` - 15% OFF up to 60.000₫).

### **Statistical Hypotheses**
- **Null Hypothesis ($H_0$):** $p_B \le p_A$ (Percentage voucher does not increase conversion rate).
- **Alternative Hypothesis ($H_1$):** $p_B > p_A$ (Percentage voucher significantly increases conversion rate).
- **Significance Threshold:** $\alpha = 0.05$ (95% Confidence Level).

In [ ]:
# Step 2: Load Experiment Observations from SmartSale Coupons & Orders Log
# Experiment duration: 14 days, randomized split (50/50)

n_A = 1250  # Visitors assigned to Group A (Fixed VND)
conv_A = 112 # Successfully completed orders

n_B = 1250  # Visitors assigned to Group B (Percentage %)
conv_B = 168 # Successfully completed orders

cr_A = conv_A / n_A
cr_B = conv_B / n_B
relative_uplift = (cr_B - cr_A) / cr_A * 100

print(f"Group A (Control - Fixed VNĐ)   : {conv_A}/{n_A} converted -> Conversion Rate = {cr_A:.2%}")
print(f"Group B (Variant - Percent %)   : {conv_B}/{n_B} converted -> Conversion Rate = {cr_B:.2%}")
print(f"Observed Relative Uplift in CR  : +{relative_uplift:.2f}%")

## 2. Hypothesis Testing: Two-Proportion Z-Test
We perform a two-sided and one-sided Z-test for two independent proportions.

In [ ]:
# Step 3: Perform Two-Proportion Z-Test
counts = np.array([conv_B, conv_A])
nobs = np.array([n_B, n_A])

# One-sided test (alternative='larger' -> B > A)
z_stat, p_value = proportions_ztest(count=counts, nobs=nobs, alternative='larger')

# Calculate 95% Confidence Interval for each group
ci_A_lower, ci_A_upper = proportion_confint(conv_A, n_A, alpha=0.05, method='wilson')
ci_B_lower, ci_B_upper = proportion_confint(conv_B, n_B, alpha=0.05, method='wilson')

print("=== STATISTICAL INFERENCE RESULTS ===")
print(f"Z-Score Statistic           : {z_stat:.4f}")
print(f"P-Value                     : {p_value:.6f}")
print(f"Group A 95% CI              : [{ci_A_lower:.2%}, {ci_A_upper:.2%}]")
print(f"Group B 95% CI              : [{ci_B_lower:.2%}, {ci_B_upper:.2%}]")

alpha = 0.05
if p_value < alpha:
    print(f"\n✓ REJECT NULL HYPOTHESIS: The p-value ({p_value:.6f}) is strictly less than alpha ({alpha}).")
    print("  The Percentage Discount Voucher provides a statistically significant increase in Conversion Rate.")
else:
    print(f"\n✗ FAIL TO REJECT NULL HYPOTHESIS: p-value ({p_value:.4f}) >= {alpha}.")

## 3. Visualization of Experiment Results
Visualizing the error bars and standard error distributions of the two promotion variants.

In [ ]:
# Step 4: Plotting Conversion Rate with 95% Confidence Intervals
fig, ax = plt.subplots(1, 2, figsize=(15, 5))

# Bar chart with error bars
groups = ['Group A (Fixed VNĐ)', 'Group B (Percentage %)']
cr_values = [cr_A * 100, cr_B * 100]
errors = [
    [(cr_A - ci_A_lower) * 100, (ci_A_upper - cr_A) * 100],
    [(cr_B - ci_B_lower) * 100, (ci_B_upper - cr_B) * 100]
]

colors = ['#64748B', '#10B981']
bars = ax[0].bar(groups, cr_values, color=colors, width=0.5, capsize=8, yerr=np.array(errors).T)
ax[0].set_title("Conversion Rate by Promotional Variant (with 95% CI)", fontsize=13, fontweight='bold')
ax[0].set_ylabel("Conversion Rate (%)")
ax[0].set_ylim(0, 18)

for bar in bars:
    height = bar.get_height()
    ax[0].annotate(f'{height:.2f}%',
                   xy=(bar.get_x() + bar.get_width() / 2, height),
                   xytext=(0, 14), textcoords="offset points",
                   ha='center', va='bottom', fontweight='bold', fontsize=11)

# Normal Distribution Curves of Estimators
x = np.linspace(0.06, 0.16, 500)
se_A = np.sqrt(cr_A * (1 - cr_A) / n_A)
se_B = np.sqrt(cr_B * (1 - cr_B) / n_B)

ax[1].plot(x, stats.norm.pdf(x, cr_A, se_A), label=f'Group A PDF (mean={cr_A:.2%})', color='#64748B', lw=2)
ax[1].plot(x, stats.norm.pdf(x, cr_B, se_B), label=f'Group B PDF (mean={cr_B:.2%})', color='#10B981', lw=2)
ax[1].fill_between(x, stats.norm.pdf(x, cr_A, se_A), alpha=0.2, color='#64748B')
ax[1].fill_between(x, stats.norm.pdf(x, cr_B, se_B), alpha=0.2, color='#10B981')
ax[1].set_title("Sampling Distribution of Estimators (Central Limit Theorem)", fontsize=13, fontweight='bold')
ax[1].set_xlabel("Conversion Rate Proportion")
ax[1].set_ylabel("Probability Density")
ax[1].legend()

plt.tight_layout()
plt.show()

## 4. Financial Impact & Business Recommendation

### **Key Takeaways**
1. **Statistically Significant Uplift:** Percentage discounts achieved **13.44% CR vs. 8.96% CR** ($p = 0.00018 < 0.05$).
2. **Psychological Perception:** Customers perceive "15% OFF" as higher value on mid-to-high ticket items, even when the absolute discount cap is identical.
3. **Decision Rule:** Roll out **Percentage Vouchers (with maximum discount caps)** as default promotional mechanics across all SmartSale storefront campaigns.